# Model metrics history

Shows how the filled posts models' scores change across training runs, read from each run's files in the
pipeline-resources `models/` folder: run counts, **legacy** and **cross-validated** scores (kept apart, because
they are measured differently), a text trend per score, and change against the previous run and against a
rolling baseline. Use the breach tables to choose the alert tolerances.

**Before running:** the parameters below are set, and `boto3` is available on the instance.
This notebook only reads from S3; it writes nothing back.

**Stop the notebook instance when you finish** - SageMaker bills while it is running.

## Parameters

- `MODELS_PATH` - S3 folder holding every model's runs: `<model>/<version>/<run_number>/`.
- `STORAGE_OPTIONS` - Polars S3 options; leave `None` unless the region isn't found.
- `MODEL_NAMES` - the models to report on.
- `BASELINE_RUNS` - how many previous runs (same version) the drift baseline's median covers.
- `STEP_TOLERANCE`, `DRIFT_TOLERANCE` - trial values: the relative change in the wrong direction, against
  the previous run and against the baseline, that counts as a breach. Tune them from the tables below.
- `ABSOLUTE_CHANGE_METRICS`, `ABSOLUTE_STEP_TOLERANCE`, `ABSOLUTE_DRIFT_TOLERANCE` - scores that sit near zero
  (bias, jumpiness), where a relative change swings wildly. They are judged on the plain difference against
  these trial tolerances instead.
- `KEY_METRICS` - `(name, level, metric, fold, higher_is_better)` for each tracked score, read from the pooled
  rows of `metrics.parquet`. `ABSOLUTE_VALUE_METRICS` are tracked as absolute values; `FLAG_METRICS` are only
  listed when above zero.
- `TREND_BLOCKS` - the characters of the text trend, lowest to highest.
- `METADATA_FILE`, `METRICS_FILE`, `SCORES_KEY` - the file names in each run's folder, and the key in
  `metadata.json` that holds a run's scores.
- Column and label names, including the names of derived columns, are defined once here (defaults are the
  training output's).

In [ ]:
import re

MODELS_PATH = "s3://<pipeline-resources-bucket>/models/"
STORAGE_OPTIONS = None  # e.g. {"aws_region": "eu-west-2"} if Polars can't find the region
MODEL_NAMES = [
    "care_home_model",
    "non_res_with_dormancy_model",
    "non_res_without_dormancy_model",
]
BASELINE_RUNS = 5
STEP_TOLERANCE = 0.10
DRIFT_TOLERANCE = 0.20
ABSOLUTE_STEP_TOLERANCE = 0.01
ABSOLUTE_DRIFT_TOLERANCE = 0.02
TREND_BLOCKS = "▁▂▃▄▅▆▇█"

# Names are defined once, here. Edit them if the training output changes.
assert MODELS_PATH.startswith("s3://") and "/" in MODELS_PATH.removeprefix("s3://").rstrip("/"), (
    "MODELS_PATH must be s3://<bucket>/<folder>/"
)
BUCKET, MODELS_KEY = MODELS_PATH.removeprefix("s3://").rstrip("/").split("/", 1)
MODELS_PREFIX = MODELS_KEY + "/"
RUN_PATTERN = re.escape(MODELS_PREFIX) + r"([^/]+)/([^/]+)/(\d+)/"

MODEL, VERSION, RUN, TIMESTAMP, EVALUATION = "model", "version", "run_number", "timestamp", "evaluation"
LEVEL, GROUP, FOLD, METRIC, VALUE, NAME = "level", "group", "fold", "metric", "value", "name"
HIGHER_IS_BETTER = "higher_is_better"
PATH = "path"
METADATA_FILE, METRICS_FILE, SCORES_KEY = "metadata.json", "metrics.parquet", "metrics"
POOLED, ALL_GROUPS, ALL_ROWS = "pooled", "all", "all_rows"
LEGACY, CROSS_VALIDATED = "legacy", "cross_validated"

RUNS, FIRST, LAST, MIN, MAX, TREND = "runs", "first", "last", "min", "max", "trend"
FIRST_RUN, LAST_RUN = "first_run", "last_run"
PREVIOUS, BASELINE = "previous", "baseline"
STEP_ADVERSE, DRIFT_ADVERSE = "step_adverse", "drift_adverse"
STEP_BREACH, DRIFT_BREACH = "step_breach", "drift_breach"
RUNS_COMPARED = "runs_compared"
MEDIAN_STEP_ADVERSE, MAX_STEP_ADVERSE, MAX_DRIFT_ADVERSE = "median_step_adverse", "max_step_adverse", "max_drift_adverse"
STEP_BREACHES, DRIFT_BREACHES = "step_breaches", "drift_breaches"

R2, RMSE, WAPE = "r2", "rmse", "weighted_absolute_percentage_error"
WITHIN_TEN = "proportion_of_model_predictions_within_ten"
WITHIN_TWENTY_FIVE = "proportion_of_model_predictions_within_twenty_five"
METADATA_METRICS = [R2, RMSE, WITHIN_TEN, WITHIN_TWENTY_FIVE]

KEY_METRICS = [
    ("posts_r2", "row_level_posts", R2, POOLED, True),
    ("posts_wape", "row_level_posts", WAPE, POOLED, False),
    ("posts_abs_bias", "row_level_posts", "bias", POOLED, False),
    ("posts_rmse", "row_level_posts", RMSE, POOLED, False),
    ("headline_r2", "headline_group_totals", R2, POOLED, True),
    ("headline_wape", "headline_group_totals", WAPE, POOLED, False),
    ("coverage", "coverage", "coverage", POOLED, True),
    ("jumpiness", "jumpiness", "mean_period_to_period_change", POOLED, False),
    ("scale_r2", "row_level_metadata_scale", R2, POOLED, True),
    ("scale_rmse", "row_level_metadata_scale", RMSE, POOLED, False),
    ("scale_within_10", "row_level_metadata_scale", WITHIN_TEN, POOLED, True),
    ("scale_within_25", "row_level_metadata_scale", WITHIN_TWENTY_FIVE, POOLED, True),
    ("hit_max_iter", "fit_diagnostics", "hit_max_iter", ALL_ROWS, False),
]
ABSOLUTE_VALUE_METRICS = ["posts_abs_bias"]
ABSOLUTE_CHANGE_METRICS = ["posts_abs_bias", "jumpiness"]
FLAG_METRICS = ["hit_max_iter"]

## Setup

In [ ]:
import json

import boto3
import polars as pl

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_width_chars(250)

# Versions as numbers, so 10.0.0 sorts after 7.0.1
VERSION_ORDER = pl.col(VERSION).str.split(".").list.eval(pl.element().cast(pl.Int64, strict=False))


def summarise_trend(scores_df):
    """One row per model, version and score: runs, first, last, min, max and a text trend.

    The trend draws each run (oldest first) as a block from the score's lowest value to its highest.
    A score that never changes draws as all lowest blocks. Missing and non-finite scores are left out.
    """
    score = pl.col(VALUE)
    spread = score.max() - score.min()
    position = pl.when(spread == 0).then(0.0).otherwise((score - score.min()) / spread)
    block = (
        (position * (len(TREND_BLOCKS) - 1))
        .round()
        .cast(pl.Int8)
        .replace_strict(dict(enumerate(TREND_BLOCKS)), return_dtype=pl.String)
    )

    return (
        scores_df.filter(score.is_finite())
        .sort(RUN)
        .group_by(MODEL, VERSION, NAME, maintain_order=True)
        .agg(
            pl.len().alias(RUNS),
            score.first().alias(FIRST),
            score.last().alias(LAST),
            score.min().alias(MIN),
            score.max().alias(MAX),
            block.str.join("").alias(TREND),
        )
        .sort(MODEL, VERSION_ORDER, maintain_order=True)
    )

## Find the runs

Lists every file under each model's folder, then reads each run's small `metadata.json` for its time and
its scores. A run with a `metrics.parquet` is **cross_validated**; a run without one is **legacy**, an
older run scored differently. The two are never compared with each other. A run needs its `metadata.json`:
a `metrics.parquet` without one (a save that failed part way) is ignored.

In [ ]:
s3 = boto3.client("s3")
keys = {
    obj["Key"]
    for model in MODEL_NAMES
    for page in s3.get_paginator("list_objects_v2").paginate(
        Bucket=BUCKET, Prefix=f"{MODELS_PREFIX}{model}/"
    )
    for obj in page.get("Contents", [])
}

run_rows = []
for key in sorted(keys):
    match = re.fullmatch(RUN_PATTERN + re.escape(METADATA_FILE), key)
    if match:
        metadata = json.loads(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read())
        scores = metadata.get(SCORES_KEY, {})
        has_metrics_file = key.replace(METADATA_FILE, METRICS_FILE) in keys
        run_rows.append(
            {
                MODEL: match[1],
                VERSION: match[2],
                RUN: int(match[3]),
                TIMESTAMP: metadata[TIMESTAMP],
                EVALUATION: CROSS_VALIDATED if has_metrics_file else LEGACY,
                **{metric: scores.get(metric) for metric in METADATA_METRICS},
            }
        )

assert run_rows, f"No runs found under {MODELS_PATH} for {MODEL_NAMES}"

runs_df = (
    pl.DataFrame(run_rows, infer_schema_length=None)
    .with_columns(pl.col(TIMESTAMP).str.to_datetime(time_zone="UTC").dt.truncate("1s"))
    .sort(MODEL, VERSION_ORDER, RUN)
)

## Runs per model and version

How many runs each model has, of which kind, and when the first and last ran. The run count and spacing
decide whether the rolling baseline has enough runs to work (`BASELINE_RUNS`).

In [ ]:
runs_df.group_by(MODEL, VERSION, EVALUATION).agg(
    pl.len().alias(RUNS),
    pl.col(TIMESTAMP).min().alias(FIRST_RUN),
    pl.col(TIMESTAMP).max().alias(LAST_RUN),
).sort(MODEL, VERSION_ORDER, EVALUATION)

## Legacy scores

The four scores saved in `metadata.json` by legacy runs, one row per run. Read each model and version as its
own series; do not read them against the cross-validated scores below.

In [ ]:
runs_df.filter(pl.col(EVALUATION) == LEGACY).drop(EVALUATION)

### Legacy trend

One row per model, version and legacy score: the number of runs, first, last, lowest and highest value, and a
text trend (one block per run, oldest first, from the lowest value to the highest).

In [ ]:
summarise_trend(
    runs_df.filter(pl.col(EVALUATION) == LEGACY).unpivot(
        METADATA_METRICS, index=[MODEL, VERSION, RUN], variable_name=NAME, value_name=VALUE
    )
)

## Load the cross-validated scores

Scans the `metrics.parquet` of each cross-validated run found above and keeps the pooled rows of
`KEY_METRICS`, so a small table is all that is collected: one row per run and tracked score. The version and
run come from each file's path. A NaN or infinite score is treated as missing (null). The sections below
need at least one cross-validated run, so this cell stops if there are none.

In [ ]:
key_metrics_df = pl.DataFrame(
    KEY_METRICS,
    schema=[NAME, LEVEL, METRIC, FOLD, HIGHER_IS_BETTER],
    orient="row",
)
metrics_paths = [
    f"s3://{BUCKET}/{MODELS_PREFIX}{model}/{version}/{run}/{METRICS_FILE}"
    for model, version, run in runs_df.filter(pl.col(EVALUATION) == CROSS_VALIDATED)
    .select(MODEL, VERSION, RUN)
    .iter_rows()
]
assert metrics_paths, "No cross-validated runs found"

cv_metrics_df = (
    pl.scan_parquet(metrics_paths, include_file_paths=PATH, storage_options=STORAGE_OPTIONS)
    .filter(pl.col(GROUP) == ALL_GROUPS)
    .join(key_metrics_df.lazy(), on=[LEVEL, METRIC, FOLD])
    .select(
        pl.col(MODEL),
        pl.col(PATH).str.extract(RUN_PATTERN, 2).alias(VERSION),
        pl.col(PATH).str.extract(RUN_PATTERN, 3).cast(pl.Int64).alias(RUN),
        pl.col(NAME),
        pl.col(HIGHER_IS_BETTER),
        pl.when(~pl.col(VALUE).is_finite())
        .then(None)
        .when(pl.col(NAME).is_in(ABSOLUTE_VALUE_METRICS))
        .then(pl.col(VALUE).abs())
        .otherwise(pl.col(VALUE))
        .alias(VALUE),
    )
    .collect()
)

## Cross-validated scores by run

One table per model: a row per run (oldest first within each version) and a column per tracked score.

In [ ]:
by_run_df = cv_metrics_df.join(runs_df.select(MODEL, VERSION, RUN, TIMESTAMP), on=[MODEL, VERSION, RUN])

for model_df in by_run_df.partition_by(MODEL, maintain_order=True):
    print(model_df[MODEL][0])
    print(model_df.pivot(on=NAME, index=[VERSION, RUN, TIMESTAMP], values=VALUE).sort(VERSION_ORDER, RUN))

## Cross-validated trend by score

One row per model, version and tracked score: the number of runs, first, last, lowest and highest value, and a
text trend (one block per run, oldest first, from the lowest value to the highest). Read the trend for
direction only; the breach tables below judge size. Runs with a missing score are left out of the trend.

In [ ]:
summarise_trend(cv_metrics_df)

## Change against the previous run and the baseline

For each tracked score and run, within one version: `step_adverse` is the change against the previous run and
`drift_adverse` the change against the median of the previous `BASELINE_RUNS` runs, both signed so that a
positive number is a **worse** score. Most scores use the relative change, judged against `STEP_TOLERANCE` and
`DRIFT_TOLERANCE`. Scores in `ABSOLUTE_CHANGE_METRICS` sit near zero, so they use the plain difference, judged
against `ABSOLUTE_STEP_TOLERANCE` and `ABSOLUTE_DRIFT_TOLERANCE`.

A check is empty (null, never a breach) when it has nothing to compare: the first run of a version has no
previous run, the baseline needs `BASELINE_RUNS` earlier runs that all have a score, a run with no score leaves
the next run's step check empty, and a relative change from exactly zero is undefined.

In [ ]:
group = [MODEL, VERSION, NAME]
adverse_sign = pl.when(pl.col(HIGHER_IS_BETTER)).then(-1.0).otherwise(1.0)
is_absolute = pl.col(NAME).is_in(ABSOLUTE_CHANGE_METRICS)
step_scale = pl.when(is_absolute).then(1.0).otherwise(pl.col(PREVIOUS).abs())
drift_scale = pl.when(is_absolute).then(1.0).otherwise(pl.col(BASELINE).abs())
step_tolerance = pl.when(is_absolute).then(ABSOLUTE_STEP_TOLERANCE).otherwise(STEP_TOLERANCE)
drift_tolerance = pl.when(is_absolute).then(ABSOLUTE_DRIFT_TOLERANCE).otherwise(DRIFT_TOLERANCE)

checks_df = (
    cv_metrics_df.filter(~pl.col(NAME).is_in(FLAG_METRICS))
    .sort(MODEL, VERSION, NAME, RUN)
    .with_columns(
        pl.col(VALUE).shift(1).over(group).alias(PREVIOUS),
        pl.col(VALUE)
        .shift(1)
        .rolling_median(BASELINE_RUNS, min_samples=BASELINE_RUNS)
        .over(group)
        .alias(BASELINE),
    )
    .with_columns(
        pl.when(step_scale != 0)
        .then(adverse_sign * (pl.col(VALUE) - pl.col(PREVIOUS)) / step_scale)
        .alias(STEP_ADVERSE),
        pl.when(drift_scale != 0)
        .then(adverse_sign * (pl.col(VALUE) - pl.col(BASELINE)) / drift_scale)
        .alias(DRIFT_ADVERSE),
    )
    .with_columns(
        (pl.col(STEP_ADVERSE) > step_tolerance).alias(STEP_BREACH),
        (pl.col(DRIFT_ADVERSE) > drift_tolerance).alias(DRIFT_BREACH),
    )
)

### Breaches

Every run that would have raised an alert at the current tolerances.

In [ ]:
checks_df.filter(pl.col(STEP_BREACH) | pl.col(DRIFT_BREACH)).select(
    MODEL, VERSION, RUN, NAME, VALUE, PREVIOUS, BASELINE, STEP_ADVERSE, DRIFT_ADVERSE
).sort(MODEL, VERSION_ORDER, RUN, NAME)

### Spread of changes

Per model, version and score: how many runs were compared, the typical (`median`) and worst (`max`) adverse
change, and how many breaches each tolerance gave. Changes are relative, or absolute for
`ABSOLUTE_CHANGE_METRICS`. Choose tolerances above the usual run-to-run spread but below the changes that
matter.

In [ ]:
checks_df.group_by(MODEL, VERSION, NAME).agg(
    pl.col(STEP_ADVERSE).count().alias(RUNS_COMPARED),
    pl.col(STEP_ADVERSE).median().alias(MEDIAN_STEP_ADVERSE),
    pl.col(STEP_ADVERSE).max().alias(MAX_STEP_ADVERSE),
    pl.col(DRIFT_ADVERSE).max().alias(MAX_DRIFT_ADVERSE),
    pl.col(STEP_BREACH).sum().alias(STEP_BREACHES),
    pl.col(DRIFT_BREACH).sum().alias(DRIFT_BREACHES),
).sort(MODEL, VERSION_ORDER, NAME)

## Fits that hit the iteration limit

Runs where a `FLAG_METRICS` score is above zero. For `hit_max_iter`, those fits stopped at the iteration
limit before converging.

In [ ]:
cv_metrics_df.filter(pl.col(NAME).is_in(FLAG_METRICS) & (pl.col(VALUE) > 0)).sort(MODEL, VERSION_ORDER, RUN).drop(HIGHER_IS_BETTER)